# Instalando pacotes

In [9]:
from io import BytesIO
import sys
!{sys.executable} -m pip install fsspec s3fs oci ocifs 
!{sys.executable} -m pip install pandas numpy==1.24.3
!{sys.executable} -m pip uninstall pyarrow -y
!{sys.executable} -m pip install pyarrow==12.0.1

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Found existing installation: pyarrow 12.0.1
Uninstalling pyarrow-12.0.1:
  Successfully uninstalled pyarrow-12.0.1
Defaulting to user installation because normal site-packages is not writeable
  Using cached pyarrow-12.0.1-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (39.0 MB)


# Carregando pacotes

In [10]:
import oci
import ocifs
import pandas as pd
import sys
import os
import pyarrow
# Funcoes customizadas
import configs.function_basic as funcoes

# Conexão ao repositório via OCI

In [11]:
#Buckets e nomes de saída nuvem = "oci://"
namespace = "@grxzqsiaote6/"
pasta_in = 'Feature_store/book_variaveis_01.parquet'
pasta_in_trusted = 'base_dados_cadastrais/'
pasta_out = 'Feature_store/'

bucket_book = f"oci://BOOKS_VARIAVEIS{namespace}{pasta_in}" 
bucket_trusted = f"oci://TRUSTED{namespace}{pasta_in_trusted}"
bucket_feature_store = f"oci://BOOKS_VARIAVEIS{namespace}{pasta_out}"

# Carregando databases

## Book_01

In [12]:
from ocifs import OCIFileSystem

fs = OCIFileSystem(config="~/.oci/config")

#files = fs.ls("oci://BOOKS_VARIAVEIS@grxzqsiaote6/Feature_store/book_variaveis_01.parquet'")
files = fs.ls(bucket_book)

print(files)

['BOOKS_VARIAVEIS@grxzqsiaote6/Feature_store/book_variaveis_01.parquet']


In [13]:
# Carregando book_01
df_book_01 = pd.read_parquet(
    bucket_book, #"oci://BOOKS_VARIAVEIS@grxzqsiaote6/Feature_store/book_variaveis_01.parquet'",
    storage_options={"config": "~/.oci/config"})
#print("Book 01 data shape:", df_book_01.shape)


In [14]:
df_book_01.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3795310 entries, 0 to 3795309
Data columns (total 16 columns):
 #   Column              Dtype   
---  ------              -----   
 0   ts_proc             object  
 1   Ano                 Int32   
 2   Mes                 Int8    
 3   FLAG_INSTALACAO     boolean 
 4   ProductDescription  string  
 5   ProductMigration    string  
 6   SCORE_01            float32 
 7   SCORE_02            float32 
 8   FPD                 Int32   
 9   NUM_CPF             string  
 10  ts_proc_partition   category
 11  SAFRA               Int32   
 12  SCORE_RATE          float32 
 13  SCORE_AVG           float32 
 14  SCORE_DIFF          float32 
 15  SCORE_MIN           float32 
dtypes: Int32(3), Int8(1), boolean(1), category(1), float32(6), object(1), string(3)
memory usage: 275.1+ MB


#### Base Dados Cadastrais

In [15]:
fs = OCIFileSystem(config="~/.oci/config")

#files = fs.ls("oci://TRUSTED@grxzqsiaote6/base_dados_cadastrais/")
files = fs.ls(bucket_trusted)

#print(files)

In [16]:
## Carregando todos arquivos em parquet de uma pasta
df_dados_cadastrais = pd.read_parquet(
    bucket_trusted, #"oci://TRUSTED@grxzqsiaote6/base_dados_cadastrais/",
    storage_options={"config": "~/.oci/config"})

print('Base Dados Cadastrais data shape:', df_dados_cadastrais.shape)

Base Dados Cadastrais data shape: (3900378, 37)


In [17]:
df_dados_cadastrais.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900378 entries, 0 to 3900377
Data columns (total 37 columns):
 #   Column              Dtype   
---  ------              -----   
 0   NUM_CPF             object  
 1   ts_proc             object  
 2   SAFRA_ANO           int32   
 3   SAFRA_MES           int32   
 4   FLAG_INSTALACAO     bool    
 5   FPD                 object  
 6   PROD                object  
 7   ProductMigration    object  
 8   STATUSRF            object  
 9   DATA_DE_NASCIMENTO  object  
 10  var_12              object  
 11  var_02              float64 
 12  var_03              float64 
 13  var_04              float64 
 14  var_05              float64 
 15  var_06              float64 
 16  var_07              float32 
 17  var_08              float64 
 18  var_09              float64 
 19  Profissao           object  
 20  var_11              float32 
 21  var_13              object  
 22  var_14              float64 
 23  Estado              object  
 24

#### Merge dos Datasets

In [18]:
# Primeiro vamos transformar a coluna SAFRA para o mesmo formato de book_01
df_dados_cadastrais['SAFRA'] = df_dados_cadastrais['SAFRA'].astype('int64')

In [19]:
cols_to_drop = [
    col for col in df_dados_cadastrais.columns
    if col in df_book_01.columns
    and col not in ['SAFRA', 'NUM_CPF']
]

df_dados_cadastrais_clean = df_dados_cadastrais.drop(columns=cols_to_drop)

df_book_02 = pd.merge(
    df_book_01,
    df_dados_cadastrais_clean,
    how='left',
    on=['SAFRA', 'NUM_CPF']
)

In [20]:
# Sanity check
df_book_01.shape[0] == df_book_02.shape[0]

True

### Feature Engineer


##### Bloco 01
Primeiro iremos aplicar duas regras de negócio:
- Utilizaremos apenas registros que possuam `STATUSRF` = *REGULAR*
- Apenas CPFs que possuam a partir de 18 anos na data de contratação do plano
    - `SAFRA` - `DATADENASCIMENTO` >= 18

##### Bloco 02
Após a remoção dos valores acima, o próximo sanity check foi limpar colunas com alto indice de nulos e cardinalidade igual a 1

##### Bloco 03
Ajustes de colunas:

- Criar regiões a partir de `CEP_3_digitos` 


- As colunas a seguir são flags, possuem apenas um valor possível, sendo ele preenchido para casos positivos e null em casos negativos
    - `var_19` e `var_21` : estar iremos transformar em 0 e 1 e mudar o nome da coluna
    - A coluna `var_25` possui a concatenação das duas colunas acima


#### Bloco 01

In [21]:
# Filtrando CPFs que possuam STATUSRF = REGULAR
df_book_02 = df_book_02[df_book_02['STATUSRF'] == 'REGULAR']

In [22]:
df_book_02['STATUSRF'].value_counts()

STATUSRF
REGULAR    3744894
Name: count, dtype: int64

In [23]:
# Transformando coluna DATADENASCIMENTO em datetime
df_book_02['DATA_DE_NASCIMENTO'] = pd.to_datetime(df_book_02['DATA_DE_NASCIMENTO'], format='%d/%m/%Y')

# Criando uma coluna DATA_SAFRA a partir da coluna SAFRA
df_book_02['DATA_SAFRA'] = pd.to_datetime(df_book_02['SAFRA'].astype(str), format='%Y%m')

# Calculando a idade a partir da data de contratação da SAFRA
df_book_02 = funcoes.calcular_idade_df(df_book_02, 'DATA_DE_NASCIMENTO', 'DATA_SAFRA', 'IDADE')

# Selecionando apenas CPFs com idade maior que 18 anos
df_book_02 = df_book_02[df_book_02['IDADE'] >= 18]

In [24]:
df_book_02['IDADE'].describe()

count    3734429.0
mean     42.063054
std      14.805385
min           18.0
25%           30.0
50%           40.0
75%           52.0
max          125.0
Name: IDADE, dtype: Float64

In [25]:
df_book_02

,ts_proc,Ano,Mes,FLAG_INSTALACAO,ProductDescription,ProductMigration,SCORE_01,SCORE_02,FPD,NUM_CPF,...,var_19,var_20,var_21,Cargo,var_23,var_24,Tipo_de_Auxilio,CEP_3_digitos,DATA_SAFRA,IDADE
0,20260309213950,2024,10,True,CMV,Aquisição,2.0,1.0,1,ZZZZZZZX7T9,...,None,None,FUNC_PRIVADO,None,None,ADMITIDO,FUNC_PRIVADO,None,2024-10-01,41
1,20260309213950,2024,10,False,CMV,<NA>,562.0,559.0,<NA>,ZZZZZZZ8TZ8,...,AUX_EMRG,None,FUNC_PRIVADO,None,None,ADMITIDO,APOSENTADO AUX_EMRG FUNC_PRIVADO,795,2024-10-01,48
2,20260309213950,2024,10,False,CMV,<NA>,585.0,559.0,<NA>,ZZZZZZW9XWN,...,None,None,FUNC_PRIVADO,None,None,ADMITIDO,FUNC_PRIVADO,798,2024-10-01,40
3,20260309213950,2024,10,True,CMV,PRE,562.0,636.0,0,ZZZZZX7XWY8,...,None,None,FUNC_PRIVADO,None,None,ADMITIDO,FUNC_PRIVADO,479,2024-10-01,40
4,20260309213950,2024,10,True,CMV,Aquisição,538.0,570.0,1,ZZZZZX8TTUZ,...,None,None,FUNC_PRIVADO,None,None,DISPENSADO,APOSENTADO FUNC_PRIVADO,398,2024-10-01,44
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3795305,20260309213950,2025,3,True,CMV,PRE,616.0,630.0,0,9999888YYU9,...,AUX_EMRG,None,FUNC_PRIVADO,EMPR/DIRETOR,None,ADMITIDO,AUX_EMRG FUNC_PRIVADO EMPR/DIRETOR,769,2025-03-01,36
3795306,20260309213950,2025,3,True,CMV,PRE,627.0,649.0,0,9999889ZN9X,...,AUX_EMRG,None,FUNC_PRIVADO,None,BOLSA_FAMILIA,ADMITIDO,AUX_EMRG FUNC_PRIVADO BOLSA_FAMILIA,339,2025-03-01,61
3795307,20260309213950,2025,3,True,CMV,Aquisição,561.0,661.0,0,999997YY7N8,...,AUX_EMRG,None,FUNC_PRIVADO,EMPR/DIRETOR,None,DISPENSADO,AUX_EMRG FUNC_PRIVADO EMPR/DIRETOR,247,2025-03-01,65
3795308,20260309213950,2025,3,True,CMV,PRE,577.0,611.0,1,999998YZYNW,...,AUX_EMRG,None,FUNC_PRIVADO,None,None,ADMITIDO,AUX_EMRG FUNC_PRIVADO,768,2025-03-01,38


#### Bloco 02

Nesta seção iremos transformar as colunas de var_18 até var_24 em binárias, visto que o comportamento delas são sociais.

In [26]:
# Aplicando a funcao para transformar as colunas em booleanas
colunas_booleanas = ['var_18', 'var_19', 'var_20', 'var_21', 'var_23', 'var_24']

df_book_02 = funcoes.criar_flag_com_nome_do_valor(df_book_02,colunas_booleanas)

In [27]:
df_book_02

,ts_proc,Ano,Mes,FLAG_INSTALACAO,ProductDescription,ProductMigration,SCORE_01,SCORE_02,FPD,NUM_CPF,...,CEP_3_digitos,DATA_SAFRA,IDADE,APOSENTADO,AUX_EMRG,FUNC_PUBL,FUNC_PRIVADO,BOLSA_FAMILIA,ADMITIDO,DISPENSADO
0,20260309213950,2024,10,True,CMV,Aquisição,2.0,1.0,1,ZZZZZZZX7T9,...,None,2024-10-01,41,0,0,0,1,0,1,0
1,20260309213950,2024,10,False,CMV,<NA>,562.0,559.0,<NA>,ZZZZZZZ8TZ8,...,795,2024-10-01,48,1,1,0,1,0,1,0
2,20260309213950,2024,10,False,CMV,<NA>,585.0,559.0,<NA>,ZZZZZZW9XWN,...,798,2024-10-01,40,0,0,0,1,0,1,0
3,20260309213950,2024,10,True,CMV,PRE,562.0,636.0,0,ZZZZZX7XWY8,...,479,2024-10-01,40,0,0,0,1,0,1,0
4,20260309213950,2024,10,True,CMV,Aquisição,538.0,570.0,1,ZZZZZX8TTUZ,...,398,2024-10-01,44,1,0,0,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3795305,20260309213950,2025,3,True,CMV,PRE,616.0,630.0,0,9999888YYU9,...,769,2025-03-01,36,0,1,0,1,0,1,0
3795306,20260309213950,2025,3,True,CMV,PRE,627.0,649.0,0,9999889ZN9X,...,339,2025-03-01,61,0,1,0,1,1,1,0
3795307,20260309213950,2025,3,True,CMV,Aquisição,561.0,661.0,0,999997YY7N8,...,247,2025-03-01,65,0,1,0,1,0,0,1
3795308,20260309213950,2025,3,True,CMV,PRE,577.0,611.0,1,999998YZYNW,...,768,2025-03-01,38,0,1,0,1,0,1,0


Nesta seção iremos lidar com valores ausentes e valores continuos
- Deleção de colunas que possuam o limiar de 70% de valores nulos
- Deleção de colunas com cardinalidade igual a 1

In [28]:
funcoes.generate_metadata(df_book_02)

,nome_variavel,tipo,qt_nulos,percent_nulos,cardinalidade
0,var_11,float32,3679339,98.52,15921
1,Profissao,object,3675305,98.42,3314
2,var_02,float64,3528001,94.47,2124
3,var_14,float64,3383444,90.60,49
4,Cargo,object,3383444,90.60,1
5,var_13,object,3170281,84.89,1502
6,var_17,float64,3163809,84.72,20
7,Estado,object,3163809,84.72,27
8,var_16,float64,3163809,84.72,1511
9,var_07,float32,3110448,83.29,223099


In [29]:
# Deletando colunas que possuam mais que 70% de valores faltantes
threshold = 0.7
df_book_02 = funcoes.drop_columns_high_missing(df_book_02, threshold)

🧹 Colunas removidas (> 70% missing): 12
 - var_02: 94.5%
 - var_06: 80.9%
 - var_07: 83.3%
 - var_08: 80.9%
 - Profissao: 98.4%
 - var_11: 98.5%
 - var_13: 84.9%
 - var_14: 90.6%
 - Estado: 84.7%
 - var_16: 84.7%
 - var_17: 84.7%
 - Cargo: 90.6%


In [30]:
# Agora iremos deletar colunas que possuem cardinalidade igual a 1
df_book_02 = funcoes.drop_single_cardinality_columns(df_book_02)

🧹 Colunas removidas (cardinalidade = 1): 5
 - ts_proc
 - ProductDescription
 - ts_proc_partition
 - PROD
 - STATUSRF


In [31]:
# Conferindo metadados
funcoes.generate_metadata(df_book_02)

,nome_variavel,tipo,qt_nulos,percent_nulos,cardinalidade
0,var_09,float64,2109710,56.49,17
1,var_12,object,1421177,38.06,15450
2,FPD,Int32,1128543,30.22,2
3,ProductMigration,string[python],1128543,30.22,3
4,Tipo_de_Auxilio,object,428223,11.47,63
5,CEP_3_digitos,object,261441,7.00,922
6,var_03,float64,242169,6.48,100
7,var_05,float64,169992,4.55,10
8,SCORE_DIFF,float32,33581,0.90,1317
9,SCORE_AVG,float32,33581,0.90,1295


#### Bloco 03

Quebra de informação da coluna `CEP_3_digitos` em:
 - `REGIAO_POSTAL`
 - `REGIAO_POSTAL_TXT`
 - `SUB_REGIAO_POSTAL`

 Transformação das colunas `var_19`, `var_21` em booleana:
 - `var_19` -> `AUX_EMRG`: 1 para quem teve essa coluna preenchida, 0 para quem não teve
 - `var_21` -> `FUNC_PRIVADO`: 1 para quem teve essa coluna preenchida, 0 para quem não teve
 
 Criação da coluna `TEMPO_CADASTRO`
 - Esta coluna só é preenchida quando a coluna `var_21`possui preenchimento
 - Iremos inferir que é quando o cadastro foi preenchido e calcularemos a nova coluna
 - Após a criação desta coluna foi realizado o drop da coluna `var_12`

In [32]:
# Criando colunas a partir de CEP_3_digitos
df_book_02 = funcoes.mapear_regiao_subregiao_texto(df_book_02, 'CEP_3_digitos')

In [33]:
# Verificando apenas as variaveis que iniciam com var_
df_book_02.filter(like='var_')

,var_12,var_03,var_04,var_05,var_09
0,2010-09-01,NaN,0.0,1.0,NaN
1,2018-09-19,43.0,0.0,2.0,5.0
2,2002-06-28,4.0,0.0,2.0,NaN
3,2009-04-15,33.0,0.0,2.0,NaN
4,2014-06-01,28.0,0.0,1.0,NaN
...,...,...,...,...,...
3795305,2019-08-10,29.0,0.0,3.0,9.0
3795306,2012-02-16,NaN,0.0,2.0,9.0
3795307,2010-04-01,43.0,0.0,1.0,7.0
3795308,2020-10-01,19.0,0.0,2.0,7.0


In [34]:
# Criando print de values_counts() das variaveis que começam com var_
for col in df_book_02.filter(like='var_').columns:
    print(f"Value counts for column: {col}")
    print(df_book_02[col].value_counts())
    print("\n")

Value counts for column: var_12
var_12
2019-04-01    8568
2019-10-01    8362
2020-02-03    8188
2019-08-01    8162
2019-02-01    8089
              ... 
1979-03-31       1
1984-06-02       1
1976-09-28       1
1984-06-24       1
1977-06-24       1
Name: count, Length: 15450, dtype: int64


Value counts for column: var_03
var_03
33.0    1146141
1.0      252882
3.0      159897
50.0     102719
17.0      97796
         ...   
68.0        857
72.0        707
66.0        489
54.0        450
74.0        388
Name: count, Length: 100, dtype: int64


Value counts for column: var_04
var_04
0.0    3338279
1.0     235215
2.0      94963
3.0      37809
4.0      15281
5.0      12882
Name: count, dtype: int64


Value counts for column: var_05
var_05
1.0     1605216
2.0     1306549
3.0      290597
4.0      178575
5.0       98917
6.0       29081
7.0       25552
9.0       15437
8.0       13813
10.0        700
Name: count, dtype: int64


Value counts for column: var_09
var_09
9.0     929711
8.0     231128


As variaveis `var_03`, `var_04`, `var_05`, `var_09` representam valores numericos continuos.

Iremos criar uma nova coluna de tempo de admissao ou dispensa

In [35]:
# Transformando a coluna var_12 em datetime
df_book_02['var_12'] = pd.to_datetime(df_book_02['var_12'], format='%d/%m/%Y')

# Criando tempo de cadastro
df_book_02 = funcoes.calcular_idade_df(df_book_02, 'var_12', 'DATA_SAFRA', 'TEMPO_CADASTRO')

In [36]:
df_book_02['TEMPO_CADASTRO'].describe()

count    2313252.0
mean     10.008105
std        6.85498
min            3.0
25%            5.0
50%            8.0
75%           12.0
max           69.0
Name: TEMPO_CADASTRO, dtype: Float64

In [37]:
# Agora iremos realizar o drop da coluna var_12
df_book_02 = df_book_02.drop(columns=['var_12'])

In [38]:
df_book_02

,Ano,Mes,FLAG_INSTALACAO,ProductMigration,SCORE_01,SCORE_02,FPD,NUM_CPF,SAFRA,SCORE_RATE,...,AUX_EMRG,FUNC_PUBL,FUNC_PRIVADO,BOLSA_FAMILIA,ADMITIDO,DISPENSADO,REGIAO_POSTAL,SUB_REGIAO_POSTAL,REGIAO_POSTAL_TXT,TEMPO_CADASTRO
0,2024,10,True,Aquisição,2.0,1.0,1,ZZZZZZZX7T9,202410,0.500000,...,0,0,1,0,1,0,Desconhecido,Desconhecido,Desconhecido,14
1,2024,10,False,<NA>,562.0,559.0,<NA>,ZZZZZZZ8TZ8,202410,0.994662,...,1,0,1,0,1,0,7,79,Centro-Oeste e Norte,6
2,2024,10,False,<NA>,585.0,559.0,<NA>,ZZZZZZW9XWN,202410,0.955556,...,0,0,1,0,1,0,7,79,Centro-Oeste e Norte,22
3,2024,10,True,PRE,562.0,636.0,0,ZZZZZX7XWY8,202410,1.131673,...,0,0,1,0,1,0,4,47,Nordeste - Bahia/Sergipe,15
4,2024,10,True,Aquisição,538.0,570.0,1,ZZZZZX8TTUZ,202410,1.059480,...,0,0,1,0,0,1,3,39,MG,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3795305,2025,3,True,PRE,616.0,630.0,0,9999888YYU9,202503,1.022727,...,1,0,1,0,1,0,7,76,Centro-Oeste e Norte,5
3795306,2025,3,True,PRE,627.0,649.0,0,9999889ZN9X,202503,1.035088,...,1,0,1,1,1,0,3,33,MG,13
3795307,2025,3,True,Aquisição,561.0,661.0,0,999997YY7N8,202503,1.178253,...,1,0,1,0,0,1,2,24,RJ e ES,14
3795308,2025,3,True,PRE,577.0,611.0,1,999998YZYNW,202503,1.058926,...,1,0,1,0,1,0,7,76,Centro-Oeste e Norte,4


In [39]:
funcoes.generate_metadata(df_book_02)

,nome_variavel,tipo,qt_nulos,percent_nulos,cardinalidade
0,var_09,float64,2109710,56.49,17
1,TEMPO_CADASTRO,Int64,1421177,38.06,67
2,ProductMigration,string[python],1128543,30.22,3
3,FPD,Int32,1128543,30.22,2
4,Tipo_de_Auxilio,object,428223,11.47,63
5,CEP_3_digitos,object,261441,7.00,922
6,var_03,float64,242169,6.48,100
7,var_05,float64,169992,4.55,10
8,SCORE_AVG,float32,33581,0.90,1295
9,SCORE_DIFF,float32,33581,0.90,1317


In [40]:
df_book_02

,Ano,Mes,FLAG_INSTALACAO,ProductMigration,SCORE_01,SCORE_02,FPD,NUM_CPF,SAFRA,SCORE_RATE,...,AUX_EMRG,FUNC_PUBL,FUNC_PRIVADO,BOLSA_FAMILIA,ADMITIDO,DISPENSADO,REGIAO_POSTAL,SUB_REGIAO_POSTAL,REGIAO_POSTAL_TXT,TEMPO_CADASTRO
0,2024,10,True,Aquisição,2.0,1.0,1,ZZZZZZZX7T9,202410,0.500000,...,0,0,1,0,1,0,Desconhecido,Desconhecido,Desconhecido,14
1,2024,10,False,<NA>,562.0,559.0,<NA>,ZZZZZZZ8TZ8,202410,0.994662,...,1,0,1,0,1,0,7,79,Centro-Oeste e Norte,6
2,2024,10,False,<NA>,585.0,559.0,<NA>,ZZZZZZW9XWN,202410,0.955556,...,0,0,1,0,1,0,7,79,Centro-Oeste e Norte,22
3,2024,10,True,PRE,562.0,636.0,0,ZZZZZX7XWY8,202410,1.131673,...,0,0,1,0,1,0,4,47,Nordeste - Bahia/Sergipe,15
4,2024,10,True,Aquisição,538.0,570.0,1,ZZZZZX8TTUZ,202410,1.059480,...,0,0,1,0,0,1,3,39,MG,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3795305,2025,3,True,PRE,616.0,630.0,0,9999888YYU9,202503,1.022727,...,1,0,1,0,1,0,7,76,Centro-Oeste e Norte,5
3795306,2025,3,True,PRE,627.0,649.0,0,9999889ZN9X,202503,1.035088,...,1,0,1,1,1,0,3,33,MG,13
3795307,2025,3,True,Aquisição,561.0,661.0,0,999997YY7N8,202503,1.178253,...,1,0,1,0,0,1,2,24,RJ e ES,14
3795308,2025,3,True,PRE,577.0,611.0,1,999998YZYNW,202503,1.058926,...,1,0,1,0,1,0,7,76,Centro-Oeste e Norte,4


#### Ajustando os tipos de dados

Durante o processo inteiro os tipos de dados foram criadas da forma correta.

Apesar de serem valores de tipos numericos, as variaveis abaixo foram mantidas como objeto pelo tipo de dado inserido que se comportam como categóricos:
 - `var_02`
 - `var_04`
 - `var_05`
 - `var_09`

In [41]:
fs = OCIFileSystem(config="~/.oci/config")

#files = fs.ls("oci://BOOKS_VARIAVEIS@grxzqsiaote6/Feature_store/book_variaveis_01.parquet'")
files = fs.ls(bucket_feature_store)

In [42]:
# Criando novo dataset
book_variaveis_02 = df_book_02.copy()

In [45]:
# Salvando o dataframe em parquet
book_variaveis_02.to_parquet(
    f"{bucket_feature_store}book_variaveis_02.parquet",
    engine="pyarrow",
    compression="snappy",
#    partition_cols=["SAFRA"],
    index=False,
    storage_options={"config": "~/.oci/config"}
)